In [1]:
import os
import zipfile
import urllib.request
import random

import numpy as np
import pandas as pd
import librosa

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoFeatureExtractor, ASTForAudioClassification
from sklearn.metrics import f1_score, classification_report

from tqdm.auto import tqdm

/home/mlguest/miniforge3/envs/torch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

SAMPLE_RATE = 16000
SINGLE_DURATION = 5
TOTAL_DURATION = SINGLE_DURATION * 2
SINGLE_LENGTH = SAMPLE_RATE * SINGLE_DURATION
TOTAL_LENGTH = SAMPLE_RATE * TOTAL_DURATION

BATCH_SIZE = 8
EPOCHS = 5
LR_HEAD = 1e-4
LR_ENCODER = 1e-5
MODEL_NAME = "MIT/ast-finetuned-audioset-10-10-0.4593"

Device: cuda


In [3]:
zip_path = "./ESC-50-master.zip"
extracted_dir = "./ESC-50-master"

if not os.path.exists(extracted_dir):
    print("Скачивание датасета...")
    urllib.request.urlretrieve(
        "https://github.com/karolpiczak/ESC-50/archive/master.zip", zip_path
    )
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("./")

df = pd.read_csv(os.path.join(extracted_dir, "meta", "esc50.csv"))
esc10_df = df[df["esc10"] == True].copy().reset_index(drop=True)

CLASSES = [
    "dog", "rain", "sea_waves", "crying_baby", "clock_tick",
    "sneezing", "helicopter", "chainsaw", "rooster", "crackling_fire",
]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)
AUDIO_DIR = os.path.join(extracted_dir, "audio")

esc10_df["label"] = esc10_df["category"].map(CLASS_TO_IDX)
print(f"ESC-10: {len(esc10_df)} записей, {NUM_CLASSES} классов")

ESC-10: 400 записей, 10 классов


In [4]:
def load_audio(filename):
    audio, _ = librosa.load(os.path.join(AUDIO_DIR, filename), sr=SAMPLE_RATE, mono=True)
    if len(audio) < SINGLE_LENGTH:
        audio = np.pad(audio, (0, SINGLE_LENGTH - len(audio)))
    else:
        audio = audio[:SINGLE_LENGTH]
    return audio.astype(np.float32)

X_wave = np.stack([
    load_audio(fn) for fn in tqdm(esc10_df["filename"], desc="Загрузка аудио")
])
y = esc10_df["label"].to_numpy(dtype=np.int64)

folds = esc10_df["fold"].to_numpy()
train_idx = np.where(np.isin(folds, [1, 2, 3]))[0]
val_idx = np.where(folds == 4)[0]
test_idx = np.where(folds == 5)[0]
print(f"Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

Загрузка аудио: 100%|██████████| 400/400 [00:00<00:00, 559.83it/s]

Train: 240 | Val: 80 | Test: 80


In [5]:
def create_sequential_mixtures(waveforms, labels, indices, n_mixtures, seed=42):
    rng = np.random.default_rng(seed)
    available_classes = np.unique(labels[indices])
    class_indices = {cls: indices[labels[indices] == cls] for cls in available_classes}

    mixtures, targets = [], []
    for _ in tqdm(range(n_mixtures), desc=f"Создание {n_mixtures} смесей"):
        selected_classes = rng.choice(available_classes, size=2, replace=False)
        selected_indices = [rng.choice(class_indices[c]) for c in selected_classes]

        combined = np.concatenate([waveforms[i] for i in selected_indices]).astype(np.float32)
        target = np.zeros(NUM_CLASSES, dtype=np.float32)
        target[selected_classes] = 1.0

        mixtures.append(combined)
        targets.append(target)

    return np.stack(mixtures), np.stack(targets)

X_train, y_train = create_sequential_mixtures(X_wave, y, train_idx, 1500, seed=42)
X_val, y_val = create_sequential_mixtures(X_wave, y, val_idx, 300, seed=43)
X_test, y_test = create_sequential_mixtures(X_wave, y, test_idx, 300, seed=44)

Создание 300 смесей: 100%|██████████| 300/300 [00:00<00:00, 4380.96it/s]


In [6]:
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)

class AudioDataset(Dataset):
    def __init__(self, waveforms, labels):
        self.waveforms = waveforms
        self.labels = labels

    def __len__(self):
        return len(self.waveforms)

    def __getitem__(self, idx):
        inputs = feature_extractor(
            self.waveforms[idx],
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt",
            padding="max_length",
            max_length=1024,
            truncation=True,
        )
        return (
            inputs["input_values"].squeeze(0),
            torch.tensor(self.labels[idx], dtype=torch.float32),
        )

train_loader = DataLoader(AudioDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(AudioDataset(X_val, y_val), batch_size=BATCH_SIZE)
test_loader = DataLoader(AudioDataset(X_test, y_test), batch_size=BATCH_SIZE)

model = ASTForAudioClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,
).to(DEVICE)

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
for p in model.audio_spectrogram_transformer.parameters():
    p.requires_grad = False

encoder_layers = model.audio_spectrogram_transformer.encoder.layer
for layer in encoder_layers[-2:]:
    for p in layer.parameters():
        p.requires_grad = True

for p in model.classifier.parameters():
    p.requires_grad = True

head_ids = {id(p) for p in model.classifier.parameters()}

head_params = [p for p in model.parameters() if p.requires_grad and id(p) in head_ids]
encoder_params = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]

print(f"Обучаемых параметров головы: {len(head_params)}")
print(f"Обучаемых параметров энкодера: {len(encoder_params)}")

optimizer = torch.optim.AdamW([
    {"params": head_params, "lr": LR_HEAD},
    {"params": encoder_params, "lr": LR_ENCODER},
])

criterion = nn.BCEWithLogitsLoss()

Обучаемых параметров головы: 4
Обучаемых параметров энкодера: 32


In [8]:
best_f1 = 0.0
patience = 4
bad = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for x, y_batch in tqdm(train_loader, desc=f"Train {epoch+1}/{EPOCHS}"):
        x, y_batch = x.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x).logits
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)

    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for x, y_batch in val_loader:
            x = x.to(DEVICE)
            probs = torch.sigmoid(model(x).logits).cpu().numpy()
            val_preds.append(probs >= 0.5)
            val_labels.append(y_batch.numpy())

    val_preds = np.vstack(val_preds).astype(int)
    val_labels = np.vstack(val_labels).astype(int)
    f1 = f1_score(val_labels, val_preds, average="micro", zero_division=0)
    print(f"Epoch {epoch+1}: loss={total_loss/len(train_loader.dataset):.4f}, val Micro F1={f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "best_ast_sequential.pt")
        bad = 0
    else:
        bad += 1
        if bad >= patience:
            print("Early stopping.")
            break

Train 1/5: 100%|██████████| 188/188 [00:32<00:00,  5.81it/s]


Epoch 1: loss=0.2218, val Micro F1=0.9540


Train 2/5: 100%|██████████| 188/188 [00:32<00:00,  5.87it/s]


Epoch 2: loss=0.0510, val Micro F1=0.9721


Train 3/5: 100%|██████████| 188/188 [00:32<00:00,  5.85it/s]


Epoch 3: loss=0.0235, val Micro F1=0.9722


Train 4/5: 100%|██████████| 188/188 [00:32<00:00,  5.86it/s]


Epoch 4: loss=0.0134, val Micro F1=0.9747


Train 5/5: 100%|██████████| 188/188 [00:32<00:00,  5.83it/s]


Epoch 5: loss=0.0087, val Micro F1=0.9781


In [9]:
model.load_state_dict(torch.load("best_ast_sequential.pt"))
model.eval()

def get_scores(loader):
    scores, labels = [], []
    with torch.no_grad():
        for x, y_batch in loader:
            x = x.to(DEVICE)
            scores.append(torch.sigmoid(model(x).logits).cpu().numpy())
            labels.append(y_batch.numpy())
    return np.vstack(scores), np.vstack(labels)

val_scores, val_labels = get_scores(val_loader)
test_scores, test_labels = get_scores(test_loader)

best_thr, best_f1_thr = 0.5, 0.0
for thr in np.arange(0.20, 0.81, 0.05):
    preds = (val_scores >= thr).astype(int)
    f1 = f1_score(val_labels, preds, average="micro", zero_division=0)
    if f1 > best_f1_thr:
        best_f1_thr, best_thr = f1, thr

print(f"\nЛучший порог: {best_thr:.2f} (val Micro F1={best_f1_thr:.4f})")

test_preds = (test_scores >= best_thr).astype(int)

print("\n=== РЕЗУЛЬТАТЫ (Последовательные звуки) ===")
print("Micro F1:", round(f1_score(test_labels, test_preds, average="micro", zero_division=0), 4))
print("Macro F1:", round(f1_score(test_labels, test_preds, average="macro", zero_division=0), 4))
print()
print(classification_report(test_labels, test_preds, target_names=CLASSES, zero_division=0))


Лучший порог: 0.50 (val Micro F1=0.9781)

=== РЕЗУЛЬТАТЫ (Последовательные звуки) ===
Micro F1: 0.9754
Macro F1: 0.9753

                precision    recall  f1-score   support

           dog       1.00      0.98      0.99        58
          rain       0.96      1.00      0.98        68
     sea_waves       1.00      1.00      1.00        66
   crying_baby       1.00      1.00      1.00        66
    clock_tick       1.00      0.96      0.98        47
      sneezing       1.00      0.98      0.99        49
    helicopter       0.96      0.87      0.92        63
      chainsaw       1.00      0.93      0.97        61
       rooster       1.00      1.00      1.00        59
crackling_fire       1.00      0.87      0.93        63

     micro avg       0.99      0.96      0.98       600
     macro avg       0.99      0.96      0.98       600
  weighted avg       0.99      0.96      0.97       600
   samples avg       0.99      0.96      0.97       600



In [10]:
from IPython.display import Audio, display
import numpy as np

def listen_examples(
    waveforms,
    targets,
    n_examples=3,
    sample_rate=SAMPLE_RATE,
    start_index=0,
    show_scores=False,
    model=None,
    device=None,
):
    indices = range(start_index, min(start_index + n_examples, len(waveforms)))

    for i in indices:
        true_ids = np.where(targets[i] == 1)[0]
        true_names = [CLASSES[c] for c in true_ids]

        print(f"\n{'='*60}")
        print(f"Пример #{i}")
        print(f"Истинные классы: {', '.join(true_names)}")
        print(f"Длительность: {len(waveforms[i]) / sample_rate:.2f} с")

        if show_scores and model is not None:
            model.eval()
            inputs = feature_extractor(
                waveforms[i],
                sampling_rate=sample_rate,
                return_tensors="pt",
                padding="max_length",
                max_length=1024,
                truncation=True,
            )
            with torch.no_grad():
                logits = model(inputs["input_values"].to(device)).logits
                probs = torch.sigmoid(logits).cpu().numpy().squeeze(0)

            top3 = np.argsort(probs)[::-1][:3]
            print("Топ-3 предсказания модели:")
            for cls_id in top3:
                marker = "✔" if targets[i, cls_id] == 1 else " "
                print(f"   {marker} {CLASSES[cls_id]:>15s}: {probs[cls_id]:.3f}")

        display(Audio(waveforms[i], rate=sample_rate))

listen_examples(
    X_train, y_train,
    n_examples=3,
    sample_rate=SAMPLE_RATE,
    start_index=0,
    show_scores=True,
    model=model,
    device=DEVICE,
)


Пример #0
Истинные классы: dog, chainsaw
Длительность: 10.00 с
Топ-3 предсказания модели:
   ✔             dog: 0.998
   ✔        chainsaw: 0.994
          helicopter: 0.004



Пример #1
Истинные классы: dog, chainsaw
Длительность: 10.00 с
Топ-3 предсказания модели:
   ✔        chainsaw: 0.998
   ✔             dog: 0.974
           sea_waves: 0.008



Пример #2
Истинные классы: clock_tick, crackling_fire
Длительность: 10.00 с
Топ-3 предсказания модели:
   ✔  crackling_fire: 0.996
   ✔      clock_tick: 0.991
             rooster: 0.008


In [11]:
import os
import zipfile
import urllib.request
import random

import numpy as np
import pandas as pd
import librosa

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoFeatureExtractor, ASTForAudioClassification
from sklearn.metrics import f1_score, classification_report

from tqdm.auto import tqdm

In [12]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

SAMPLE_RATE = 16000
DURATION = 5
AUDIO_LENGTH = SAMPLE_RATE * DURATION

BATCH_SIZE = 8
EPOCHS = 5
LR_HEAD = 1e-4
LR_ENCODER = 1e-5
MODEL_NAME = "MIT/ast-finetuned-audioset-10-10-0.4593"

Device: cuda


In [13]:
zip_path = "./ESC-50-master.zip"
extracted_dir = "./ESC-50-master"

if not os.path.exists(extracted_dir):
    print("Скачивание датасета...")
    urllib.request.urlretrieve(
        "https://github.com/karolpiczak/ESC-50/archive/master.zip", zip_path
    )
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("./")

df = pd.read_csv(os.path.join(extracted_dir, "meta", "esc50.csv"))
esc10_df = df[df["esc10"] == True].copy().reset_index(drop=True)

CLASSES = [
    "dog", "rain", "sea_waves", "crying_baby", "clock_tick",
    "sneezing", "helicopter", "chainsaw", "rooster", "crackling_fire",
]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)
AUDIO_DIR = os.path.join(extracted_dir, "audio")

esc10_df["label"] = esc10_df["category"].map(CLASS_TO_IDX)

In [14]:
def load_audio(filename):
    audio, _ = librosa.load(os.path.join(AUDIO_DIR, filename), sr=SAMPLE_RATE, mono=True)
    if len(audio) < AUDIO_LENGTH:
        audio = np.pad(audio, (0, AUDIO_LENGTH - len(audio)))
    else:
        audio = audio[:AUDIO_LENGTH]
    return audio.astype(np.float32)

X_wave = np.stack([load_audio(fn) for fn in tqdm(esc10_df["filename"], desc="Аудио")])
y = esc10_df["label"].to_numpy(dtype=np.int64)

folds = esc10_df["fold"].to_numpy()
train_idx = np.where(np.isin(folds, [1, 2, 3]))[0]
val_idx = np.where(folds == 4)[0]
test_idx = np.where(folds == 5)[0]

Аудио: 100%|██████████| 400/400 [00:00<00:00, 721.42it/s]


In [15]:
def create_overlapping_mixtures(waveforms, labels, indices, n_mixtures, seed=42):
    rng = np.random.default_rng(seed)
    available_classes = np.unique(labels[indices])
    class_indices = {cls: indices[labels[indices] == cls] for cls in available_classes}

    mixtures, targets = [], []
    for _ in tqdm(range(n_mixtures), desc=f"Наложение {n_mixtures}"):
        selected_classes = rng.choice(available_classes, size=2, replace=False)
        selected_indices = [rng.choice(class_indices[c]) for c in selected_classes]

        mixed = np.zeros_like(waveforms[0], dtype=np.float32)
        for idx in selected_indices:
            audio = waveforms[idx].copy()
            rms = np.sqrt(np.mean(audio ** 2))
            if rms > 1e-8:
                audio = audio / rms
            gain = float(rng.uniform(0.7, 1.0))
            mixed += gain * audio

        peak = np.max(np.abs(mixed))
        if peak > 0:
            mixed = mixed / peak

        target = np.zeros(NUM_CLASSES, dtype=np.float32)
        target[selected_classes] = 1.0

        mixtures.append(mixed)
        targets.append(target)

    return np.stack(mixtures), np.stack(targets)

X_train, y_train = create_overlapping_mixtures(X_wave, y, train_idx, 1500, seed=42)
X_val, y_val = create_overlapping_mixtures(X_wave, y, val_idx, 300, seed=43)
X_test, y_test = create_overlapping_mixtures(X_wave, y, test_idx, 300, seed=44)

Наложение 300: 100%|██████████| 300/300 [00:00<00:00, 4330.27it/s]


In [16]:
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)

class AudioDataset(Dataset):
    def __init__(self, waveforms, labels):
        self.waveforms = waveforms
        self.labels = labels

    def __len__(self):
        return len(self.waveforms)

    def __getitem__(self, idx):
        inputs = feature_extractor(
            self.waveforms[idx],
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt",
            padding="max_length",
            max_length=1024,
            truncation=True,
        )
        return (
            inputs["input_values"].squeeze(0),
            torch.tensor(self.labels[idx], dtype=torch.float32),
        )

train_loader = DataLoader(AudioDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(AudioDataset(X_val, y_val), batch_size=BATCH_SIZE)
test_loader = DataLoader(AudioDataset(X_test, y_test), batch_size=BATCH_SIZE)

model = ASTForAudioClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,
).to(DEVICE)

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [17]:
for p in model.audio_spectrogram_transformer.parameters():
    p.requires_grad = False

encoder_layers = model.audio_spectrogram_transformer.encoder.layer
for layer in encoder_layers[-2:]:
    for p in layer.parameters():
        p.requires_grad = True

for p in model.classifier.parameters():
    p.requires_grad = True

head_ids = {id(p) for p in model.classifier.parameters()}
head_params = [p for p in model.parameters() if p.requires_grad and id(p) in head_ids]
encoder_params = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]

print(f"Обучаемых параметров головы: {len(head_params)}")
print(f"Обучаемых параметров энкодера: {len(encoder_params)}")

optimizer = torch.optim.AdamW([
    {"params": head_params, "lr": LR_HEAD},
    {"params": encoder_params, "lr": LR_ENCODER},
])

criterion = nn.BCEWithLogitsLoss()

Обучаемых параметров головы: 4
Обучаемых параметров энкодера: 32


In [18]:
best_f1 = 0.0
patience = 4
bad = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for x, y_batch in tqdm(train_loader, desc=f"Train {epoch+1}/{EPOCHS}"):
        x, y_batch = x.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x).logits
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)

    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for x, y_batch in val_loader:
            x = x.to(DEVICE)
            probs = torch.sigmoid(model(x).logits).cpu().numpy()
            val_preds.append(probs >= 0.5)
            val_labels.append(y_batch.numpy())

    val_preds = np.vstack(val_preds).astype(int)
    val_labels = np.vstack(val_labels).astype(int)
    f1 = f1_score(val_labels, val_preds, average="micro", zero_division=0)
    print(f"Epoch {epoch+1}: loss={total_loss/len(train_loader.dataset):.4f}, val Micro F1={f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "best_ast_overlap.pt")
        bad = 0
    else:
        bad += 1
        if bad >= patience:
            print("Early stopping.")
            break

Train 1/5: 100%|██████████| 188/188 [00:32<00:00,  5.78it/s]


Epoch 1: loss=0.2413, val Micro F1=0.9208


Train 2/5: 100%|██████████| 188/188 [00:31<00:00,  5.90it/s]


Epoch 2: loss=0.0679, val Micro F1=0.9455


Train 3/5: 100%|██████████| 188/188 [00:31<00:00,  5.89it/s]


Epoch 3: loss=0.0361, val Micro F1=0.9618


Train 4/5: 100%|██████████| 188/188 [00:31<00:00,  5.88it/s]


Epoch 4: loss=0.0229, val Micro F1=0.9596


Train 5/5: 100%|██████████| 188/188 [00:32<00:00,  5.85it/s]


Epoch 5: loss=0.0157, val Micro F1=0.9579


In [19]:
model.load_state_dict(torch.load("best_ast_overlap.pt"))
model.eval()

def get_scores(loader):
    scores, labels = [], []
    with torch.no_grad():
        for x, y_batch in loader:
            x = x.to(DEVICE)
            scores.append(torch.sigmoid(model(x).logits).cpu().numpy())
            labels.append(y_batch.numpy())
    return np.vstack(scores), np.vstack(labels)

val_scores, val_labels = get_scores(val_loader)
test_scores, test_labels = get_scores(test_loader)

best_thr, best_f1_thr = 0.5, 0.0
for thr in np.arange(0.20, 0.81, 0.05):
    preds = (val_scores >= thr).astype(int)
    f1 = f1_score(val_labels, preds, average="micro", zero_division=0)
    if f1 > best_f1_thr:
        best_f1_thr, best_thr = f1, thr

print(f"\nЛучший порог: {best_thr:.2f} (val Micro F1={best_f1_thr:.4f})")

test_preds = (test_scores >= best_thr).astype(int)

print("\n=== РЕЗУЛЬТАТЫ (Наложенные звуки) ===")
print("Micro F1:", round(f1_score(test_labels, test_preds, average="micro", zero_division=0), 4))
print("Macro F1:", round(f1_score(test_labels, test_preds, average="macro", zero_division=0), 4))
print()
print(classification_report(test_labels, test_preds, target_names=CLASSES, zero_division=0))


Лучший порог: 0.50 (val Micro F1=0.9618)

=== РЕЗУЛЬТАТЫ (Наложенные звуки) ===
Micro F1: 0.9302
Macro F1: 0.926

                precision    recall  f1-score   support

           dog       0.95      0.91      0.93        68
          rain       0.94      1.00      0.97        51
     sea_waves       1.00      0.96      0.98        51
   crying_baby       1.00      0.97      0.98        60
    clock_tick       1.00      0.91      0.95        66
      sneezing       1.00      0.92      0.96        51
    helicopter       0.84      0.50      0.63        62
      chainsaw       1.00      0.93      0.96        67
       rooster       0.98      0.97      0.98        65
crackling_fire       1.00      0.85      0.92        59

     micro avg       0.98      0.89      0.93       600
     macro avg       0.97      0.89      0.93       600
  weighted avg       0.97      0.89      0.92       600
   samples avg       0.98      0.89      0.92       600



In [20]:
from IPython.display import Audio, display
import numpy as np

def listen_examples(
    waveforms,
    targets,
    n_examples=3,
    sample_rate=SAMPLE_RATE,
    start_index=0,
    show_scores=False,
    model=None,
    device=None,
):
    indices = range(start_index, min(start_index + n_examples, len(waveforms)))

    for i in indices:
        true_ids = np.where(targets[i] == 1)[0]
        true_names = [CLASSES[c] for c in true_ids]

        print(f"\n{'='*60}")
        print(f"Пример #{i}")
        print(f"Истинные классы: {', '.join(true_names)}")
        print(f"Длительность: {len(waveforms[i]) / sample_rate:.2f} с")

        if show_scores and model is not None:
            model.eval()
            inputs = feature_extractor(
                waveforms[i],
                sampling_rate=sample_rate,
                return_tensors="pt",
                padding="max_length",
                max_length=1024,
                truncation=True,
            )
            with torch.no_grad():
                logits = model(inputs["input_values"].to(device)).logits
                probs = torch.sigmoid(logits).cpu().numpy().squeeze(0)

            top3 = np.argsort(probs)[::-1][:3]
            print("Топ-3 предсказания модели:")
            for cls_id in top3:
                marker = "✔" if targets[i, cls_id] == 1 else " "
                print(f"   {marker} {CLASSES[cls_id]:>15s}: {probs[cls_id]:.3f}")

        display(Audio(waveforms[i], rate=sample_rate))

listen_examples(
    X_train, y_train,
    n_examples=3,
    sample_rate=SAMPLE_RATE,
    start_index=0,
    show_scores=True,
    model=model,
    device=DEVICE,
)


Пример #0
Истинные классы: dog, chainsaw
Длительность: 5.00 с
Топ-3 предсказания модели:
   ✔        chainsaw: 0.991
   ✔             dog: 0.985
                rain: 0.017



Пример #1
Истинные классы: sneezing, chainsaw
Длительность: 5.00 с
Топ-3 предсказания модели:
   ✔        sneezing: 0.993
   ✔        chainsaw: 0.950
          helicopter: 0.079



Пример #2
Истинные классы: clock_tick, chainsaw
Длительность: 5.00 с
Топ-3 предсказания модели:
   ✔        chainsaw: 0.990
   ✔      clock_tick: 0.981
                rain: 0.013
